## 1. Imports

In [27]:
import pandas as pd
import joblib

from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score,
)
from sklearn.model_selection import train_test_split

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)

from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
)
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor

## 2. Load data

In [28]:
DF_PATH = Path("../data/processed/feature_engineered_df.parquet")
df = pd.read_parquet(DF_PATH)

df.head()

,property_type,room_type,accommodates,bathrooms,bed_type,cancellation_policy,cleaning_fee,city,host_has_profile_pic,host_identity_verified,...,has_wifi,has_kitchen,has_heating,description_length,description_word_count,does_host_respond,accommodates_per_bedroom,beds_per_bedroom,bathrooms_per_bedroom,log_price
0,Apartment,Entire home/apt,3,1.0,Real Bed,strict,1,NYC,t,t,...,1,1,1,211,31,0,3.000000,1.0,1.000000,5.010635
1,Apartment,Entire home/apt,7,1.0,Real Bed,strict,1,NYC,t,f,...,1,1,1,1000,172,1,2.333333,1.0,0.333333,5.129899
2,Apartment,Entire home/apt,5,1.0,Real Bed,moderate,1,NYC,t,t,...,1,1,1,1000,172,1,5.000000,3.0,1.000000,4.976734
3,House,Entire home/apt,4,1.0,Real Bed,flexible,1,SF,t,t,...,1,1,1,468,78,0,2.000000,1.0,0.500000,6.620073
4,Apartment,Entire home/apt,2,1.0,Real Bed,moderate,1,DC,t,t,...,1,1,1,699,120,1,0.000000,0.0,0.000000,4.744932


## 3. Separate features and target

In [29]:
X = df.drop(columns="log_price")
y = df["log_price"]

## 4. Train test split

In [30]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    shuffle=True,
)

## 5. Identify column types

In [31]:
categorical_columns = X_train.select_dtypes(include=["object", "string"]).columns.to_list()
numerical_columns = X_train.select_dtypes(include=["number", "bool"]).columns.to_list()

print(f"Numerical columns: ({len(numerical_columns)})")
print()
print(f"Categorical columns: ({len(categorical_columns)})")

Numerical columns: (24)

Categorical columns: (10)


## 6. Build preprocessing pipelines

In [32]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)

## 7. ColumnTransformer

In [33]:
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numerical_columns),
        ("categorical", categorical_pipeline, categorical_columns),
    ]
)

## 8. Create models

In [34]:
models = {
    "Dummy Regressor": DummyRegressor(strategy="median"),
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        random_state=42, 
        n_estimators=100,
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42,
    )
}

## 9. Train every model

In [35]:
results = []

for name, model in models.items():
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    results.append(
        {
            "Model": name,
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2,
        }
    )

results = pd.DataFrame(results).sort_values("RMSE").round(2)
results

,Model,MAE,RMSE,R2
2,Random Forest,0.28,0.39,0.71
1,Linear Regression,0.30,0.41,0.67
3,Gradient Boosting,0.30,0.41,0.67
0,Dummy Regressor,0.56,0.72,-0.01


## 10. Save the best model

The Random Forest regressor achieved the lowest RMSE among the evaluated baseline models. It is therefore selected as the baseline model and saved for future hyperparameter tuning and evaluation.

In [41]:
best_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestRegressor(
            random_state=42, 
            n_estimators=100,)),
    ]
)
best_model.fit(X_train, y_train)

MODEL_PATH = Path("../models/random_forest_baseline.pkl")
joblib.dump(best_model, MODEL_PATH)

print(f"Model saved to {MODEL_PATH}")

Model saved to ..\models\random_forest_baseline.pkl
